In [3]:
!pip install langchain_groq langchain_core langchain_community
!pip install pypdf
!pip install chromadb
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.5/305.5 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 107.8 MB/s eta 0

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature = 0,
    groq_api_key = "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",
    model_name = "llama-3.3-70b-versatile"
)
result = llm.invoke("What is data science?")
print(result.content)

**Data Science: An Overview**

Data science is a multidisciplinary field that combines elements of computer science, statistics, and domain-specific knowledge to extract insights and knowledge from data. It involves using various techniques, tools, and algorithms to analyze and interpret complex data, often to inform business decisions, solve problems, or identify opportunities.

**Key Components of Data Science**
---------------------------------

1. **Data**: The raw material for data science, which can come from various sources, such as databases, APIs, files, or sensors.
2. **Statistics and Machine Learning**: The use of statistical and machine learning techniques to analyze and model data, including regression, classification, clustering, and more.
3. **Programming**: The use of programming languages, such as Python, R, or SQL, to manipulate and analyze data.
4. **Domain Knowledge**: The understanding of the specific domain or industry being analyzed, which is essential for interp

In [5]:
!pip install gradio

In [6]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
import gradio as gr
import os

def initialize_llm():
  llm = ChatGroq(
    temperature = 0,
    groq_api_key = "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",
    model_name = "llama3-70b-8192"
)
  return llm

def create_vector_db():
  loader = DirectoryLoader("/content/data/", glob = '*.pdf',loader_cls=PyPDFLoader)
  documents = loader.load()
  text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
  texts = text_splitter.split_documents(documents)
  embeddings = HuggingFaceEmbeddings(model_name ='sentence-transformers/all-MiniLM-L6-v2')
  vector_db = Chroma.from_documents(texts, embeddings, persist_directory = './chroma_db')
  vector_db.persist()

  print("ChromaDB created and data saved")

  return vector_db

# create_vector_db()

def setup_qa_chain(vector_db, llm):
  retriever = vector_db.as_retriever()
  prompt_template = """ You are a compassionate mental health chatbot. Respond thoughtfully to the following question:
   {context}
   User:{question}
   Chatbot:"""
  PROMPT = PromptTemplate(template = prompt_template , input_variables  = ["context", "question"])
  qa_chain = RetrievalQA.from_chain_type(
      llm = llm,
      chain_type = "stuff",
      retriever = retriever,
      chain_type_kwargs = {"prompt": PROMPT},
      return_source_documents = True)
  return qa_chain


def main():
  print("Initializing( Chatbot..........")
  llm =initialize_llm()

  db_path ="/content/chroma_db"

  if not os.path.exists(db_path):
    vector_db = create_vector_db()
  else:
    embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
    vector_db = Chroma(persist_directory = db_path, embedding_function = embeddings)

  qa_chain = setup_qa_chain(vector_db, llm)

  while True:
    query = input("\nHuman:")
    if query.lower() == "exit":
      print("Chatbot: Take Care of yourself, Goodbye!")
      break
    response = qa_chain.invoke({"query":query})
    print(f"Chatbot: {response}")

  # print("Exiting Chatbot")

if __name__ == "__main__":
  main()


Initializing( Chatbot..........


/tmp/ipython-input-6-3067154661.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name ='sentence-transformers/all-MiniLM-L6-v2')
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
/tmp/ipython-input-6-3067154661.py:26: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


ChromaDB created and data saved

Human:hii


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Chatbot: {'query': 'hii', 'result': "Hello there! It's wonderful that you're exploring the concept of mental health and what constitutes a good life. The definitions you provided from various sources, such as Rowling et al. (2002) and the World Health Organization (WHO), highlight the importance of having a sense of purpose, quality relationships, self-respect, and mastery.\n\nI completely agree that these definitions are incomplete without considering the social and physical environments that influence individuals. Our surroundings, relationships, and experiences all play a significant role in shaping our mental health and well-being.\n\nIt's fascinating to see how different definitions and frameworks, such as Ryff and Singer's (1998) concept of positive human health, emphasize the importance of psychological, emotional, creative, intellectual, and spiritual development. The ability to form and maintain meaningful relationships, enjoy solitude, and develop self-awareness are all cruci

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
import os
import gradio as gr

def initialize_llm():
  llm = ChatGroq(
    temperature = 0,
    groq_api_key = "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",
    model_name = "llama-3.3-70b-versatile"
)
  return llm

def create_vector_db():
  loader = DirectoryLoader("/content/data/", glob = '*.pdf',loader_cls=PyPDFLoader)
  documents = loader.load()
  text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
  texts = text_splitter.split_documents(documents)
  embeddings = HuggingFaceEmbeddings(model_name ='sentence-transformers/all-MiniLM-L6-v2')
  vector_db = Chroma.from_documents(texts, embeddings, persist_directory = './chroma_db')
  vector_db.persist()

  print("ChromaDB created and data saved")
  return vector_db

# create_vector_db()

def setup_qa_chain(vector_db, llm):
  retriever = vector_db.as_retriever()
  prompt_template = """ You are a compassionate mental health chatbot. Respond thoughtfully to the following question:
   Context:{context}
   User:{question}
   Chatbot:"""
  PROMPT = PromptTemplate(template = prompt_template , input_variables  = ["context", "question"])

  qa_chain = RetrievalQA.from_chain_type(
      llm = llm,
      chain_type = "stuff",
      retriever = retriever,
      chain_type_kwargs = {"prompt": PROMPT},
      return_source_documents = True)
  return qa_chain



print("Initializing( Chatbot..........")
llm =initialize_llm()

db_path ="/content/chroma_db"

if not os.path.exists(db_path):
  vector_db = create_vector_db()
else:
  embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
  vector_db = Chroma(persist_directory = db_path, embedding_function = embeddings)


qa_chain = setup_qa_chain(vector_db, llm)

def chatbot_response(user_input, history=[]):
  if not user_input.strip():
    return"Please provide a valid input"
  try:
        result = qa_chain(user_input)
        response = result["result"]
        if  isinstance(result,dict):
          return response
        else:
          return response
  except Exception as e:
        return f"An error occurred: {str(e)}", history

with gr.Blocks(theme= 'Respair/Shiki@1.2.1') as app:
  chatbot = gr.ChatInterface(fn = chatbot_response, title = "Mental Health Chatbot")

  app.launch(share=True)

def cli_chat():
  while True:
    query = input("\nYou:")
    if query.lower() == "exit":
      print("Chatbot: Take Care of yourself, Goodbye!")
      break
    response = qa_chain.invoke({"query":query})
    print(f"Chatbot: {response}")

cli_chat()
#


Initializing( Chatbot..........


/tmp/ipython-input-7-4013276400.py:60: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory = db_path, embedding_function = embeddings)
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


theme_schema%401.2.1.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7bf7ff3e026d6866d3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipython-input-7-4013276400.py:69: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain(user_input)
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
